# Scissor Meachanism

This notebook models a scissor mechansim made from four rods, six hinges and two supports. as two indipendent rods joined by a hinge treated as constraint. The functions shown in the [double pendulum notebook](./double_pendulum.ipynb) are used within the [bar_joint_model class](../api.html#quivira.bar_joint_model), which simplifies the construction of such mechanisms.

In [ ]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

Initialize the model and set the gravity constant.

In [ ]:
model = qv.bar_joint_model()
model.set_g(9.81)

The mechanism is being built. There are different ways the bars are added to the model, all highlighted here.
[`add_bar`](../api.html#quivira.bar_joint_model.add_bar) just adds the minimal information, length and mass. Theoretically, the equations of motion could be built from just this information. Eventually, it needs to be followed by [`set_init_bar_attitude`](../api.html#quivira.bar_joint_model.set_init_bar_attitude) to add an initial position and rotation.
[`add_bar_at_attitude`](../api.html#quivira.bar_joint_model.add_bar_at_attitude) combines the two for simplicity.
[`add_bar_from_point_pair`](../api.html#quivira.bar_joint_model.add_bar_from_point_pair) uses the coordinates of a start and an end point.

Joints only require the indices of the bars they join together and the respective position along the bars.
For a bar with midpoint position $(x, y)$ and angle $\theta$, a point on the bar is described by
$$
\mathbf{p}(s) = \begin{bmatrix} x \\ y \end{bmatrix}
+ s\,L \begin{bmatrix} \cos\theta \\ \sin\theta \end{bmatrix},
$$
where $L$ is the bar length and $s \in [-\tfrac{1}{2}, \tfrac{1}{2}]$ is the normalized position along the bar measured from its center. The endpoints correspond to $s = -\tfrac{1}{2}$ and $s = \tfrac{1}{2}$, while intermediate locations use any value in that interval.

Supports tie a point on a bar, defined with the $s$ formulation from above, to a global position, and lock the dof in x and/or y. Thereofre, they are the only instance where both the topology and the initial state strictly have to be defined at the same time.

In [ ]:
# First two bars
model.add_bar(length=np.sqrt(4**2 + 4**2), mass=1)
model.add_bar_at_attitude(x0=2, y0=2, theta0=-np.pi/4, length=np.sqrt(4**2 + 4**2), mass=1)
model.add_joint(bar_index=[0, 1], s=[0.0, 0.0])

# Second two bars
model.add_bar_from_point_pair(start=(4, 0), end=(8, 4), mass_per_length=(1/np.sqrt(4**2 + 4**2)))
model.add_bar_from_point_pair(start=(4, 4), end=(8, 0), mass_per_length=(1/np.sqrt(4**2 + 4**2)))
model.add_joint(bar_index=[2, 3], s=[0.0, 0.0])

# Connect them
model.add_joint(bar_index=[0, 3], s=[0.5, -0.5])
model.add_joint(bar_index=[1, 2], s=[0.5, -0.5])

# Make supports
model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False, x=0, y=0)

model.add_support(bar_index=1, s=-0.5, dof_in_x=False, dof_in_y=True, x=0, y=4)

The equations of motion are found, which we then use to instantiate heyoka Taylor adaptive integrator

In [ ]:
ode_system = model.build_lagrange_eom()
ta = hy.taylor_adaptive(ode_system, compact_mode=True)

Before we can integrate, we need to complete the setup of the initial state. While we set the initial positions and rotations of most bars when we added them to the model, we still need to assign an attitude to bar 0. All velocities are set to zero by default, which we can change for each bar with [`set_init_bar_velocities`](../api.html#quivira.bar_joint_model.set_init_bar_velocities). Alternatively, we could also use [`set_initial_state`](../api.html#quivira.bar_joint_model.set_initial_state) to set all initial states at the same time and disregard what we have initialized before, as we will see later in this notebook.

To check if we are satisfied with our model, we can print it. There should be no `None` values.

In [ ]:
model.set_init_bar_attitude(bar_index=0, x0=2, y0=2, theta0=np.pi/4)
model.print_model()

We can now extract that initial state as a list that will be passed to the integrator. We then use [`check_constraint_fulfillment`](../api.html#quivira.bar_joint_model.check_constraint_fulfillment) to ensure that the initial state is possible with the topological constraints we defined in the beginning.

In [ ]:
state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

Now we can integrate, sampling 401 times and for a duration of 1.2 seconds.

In [ ]:
ta.time = 0.0
ta.state[:] = state0

t_end = 1
tgrid = np.linspace(0.0, t_end, 401)
solution = ta.propagate_grid(tgrid)[-1]

Animate the solution

In [ ]:
save_gif = False

def bar_endpoints_from_state(state, rod_length):
    x, y, theta = state[0], state[1], state[2]
    u = np.array([np.cos(theta), np.sin(theta)])
    start = np.array([x, y]) - 0.5 * rod_length * u
    end = np.array([x, y]) + 0.5 * rod_length * u
    return start, end

def point_on_bar(q, bar_index, s):
    x, y, theta = q[3 * bar_index : 3 * bar_index + 3]
    L = model.bars[bar_index].length
    return np.array([x, y]) + s * L * np.array([np.cos(theta), np.sin(theta)])

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_aspect("equal")
ax.grid(True)

def init_anim():
    ax.clear()
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 5)
    return ()

def update_anim(frame):
    ax.clear()
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 5)

    state = solution[frame]
    q = state[:len(model.q)]

    # draw bars without endpoint markers
    for i in range(len(model.bars)):
        x, y, theta = q[3 * i : 3 * i + 3]
        start, end = bar_endpoints_from_state(np.array([x, y, theta]), model.bars[i].length)
        ax.plot([start[0], end[0]], [start[1], end[1]], lw=3, color=f"C{i}", solid_capstyle="round")

    # draw hinge markers (one marker per joint)
    seen_joints = set()
    for joint in model.joints:
        p0 = point_on_bar(q, joint.bar_index[0], joint.s[0])
        key = tuple(np.round(p0, 12))
        if key in seen_joints:
            continue
        seen_joints.add(key)
        ax.plot(p0[0], p0[1], "o", color="black", ms=7, mec="black", mew=1.2)

    # draw support markers
    support_size = 0.5
    for support in model.supports:
        p = point_on_bar(q, support.bar_index, support.s)

        if (not support.dof_in_x) and (not support.dof_in_y):
            # fixed in x and y -> +
            ax.plot(p[0], p[1], "+", color="black", ms=11, mew=2.2)
        elif support.dof_in_x and (not support.dof_in_y):
            # x open: horizontal line
            ax.plot([p[0] - support_size, p[0] + support_size],
                    [p[1], p[1]],
                    color="black", lw=2)
        elif support.dof_in_y and (not support.dof_in_x):
            # y open: vertical line
            ax.plot([p[0], p[0]],
                    [p[1] - support_size, p[1] + support_size],
                    color="black", lw=2)
        else:
            # if both x and y are free, keep it simple
            ax.plot(p[0], p[1], "o", color="black", ms=4)

    return ()

anim = FuncAnimation(
    fig,
    update_anim,
    frames=len(solution),
    interval=20,
    blit=False,
    init_func=init_anim,
)
if save_gif:
    gif_path = "scissor_mechanism.gif"
    anim.save(gif_path, writer="pillow", fps=30, dpi=150)
display(HTML(anim.to_jshtml()))
plt.close(fig)

We can now extract the final state of the integration, and set it as the initial state for a new integration.

In [ ]:
final_state = solution[-1]
model.set_initial_state(final_state)
state1 = model.get_initial_state()

ta.time = 0.0
ta.state[:] = state1

t_end = 0.1
tgrid = np.linspace(0.0, t_end, 401)
solution2 = ta.propagate_grid(tgrid)[-1]

Animate again

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_aspect("equal")
ax.grid(True)

def init_anim():
    ax.clear()
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 5)
    return ()

def update_anim(frame):
    ax.clear()
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 5)

    state = solution2[frame]
    q = state[:len(model.q)]

    # draw bars without endpoint markers
    for i in range(len(model.bars)):
        x, y, theta = q[3 * i : 3 * i + 3]
        start, end = bar_endpoints_from_state(np.array([x, y, theta]), model.bars[i].length)
        ax.plot([start[0], end[0]], [start[1], end[1]], lw=3, color=f"C{i}", solid_capstyle="round")

    # draw hinge markers (one marker per joint)
    seen_joints = set()
    for joint in model.joints:
        p0 = point_on_bar(q, joint.bar_index[0], joint.s[0])
        key = tuple(np.round(p0, 12))
        if key in seen_joints:
            continue
        seen_joints.add(key)
        ax.plot(p0[0], p0[1], "o", color="black", ms=7, mec="black", mew=1.2)

    # draw support markers
    support_size = 0.5
    for support in model.supports:
        p = point_on_bar(q, support.bar_index, support.s)

        if (not support.dof_in_x) and (not support.dof_in_y):
            # fixed in x and y -> +
            ax.plot(p[0], p[1], "+", color="black", ms=11, mew=2.2)
        elif support.dof_in_x and (not support.dof_in_y):
            # x open: horizontal line
            ax.plot([p[0] - support_size, p[0] + support_size],
                    [p[1], p[1]],
                    color="black", lw=2)
        elif support.dof_in_y and (not support.dof_in_x):
            # y open: vertical line
            ax.plot([p[0], p[0]],
                    [p[1] - support_size, p[1] + support_size],
                    color="black", lw=2)
        else:
            # if both x and y are free, keep it simple
            ax.plot(p[0], p[1], "o", color="black", ms=4)

    return ()

anim = FuncAnimation(
    fig,
    update_anim,
    frames=len(solution2),
    interval=20,
    blit=False,
    init_func=init_anim,
)

display(HTML(anim.to_jshtml()))
plt.close(fig)